# Part 4 – Multi‑Label Classification

In traditional classification tasks, **each instance is assigned exactly one label** from a set of possible classes.
This is suitable for problems like predicting the type of animal in a photo or the sentiment of a review.

However, many real‑world problems are more complex — **each instance may belong to multiple classes at once**.
This is where **multi‑label classification** becomes essential.


## What is a Multi‑Label Classification Task?

Multi‑label classification is a machine learning problem in which **each sample is associated with a set of labels**, rather than a single label.

Unlike *multi‑class* classification, labels are **not mutually exclusive**.

### Examples
- A news article tagged with *politics*, *economy*, and *international*
- A song classified as *rock* and *alternative*
- A product labeled *eco‑friendly*, *organic*, and *vegan*


## Challenges of Multi‑Label Classification

Multi‑label learning introduces several challenges:

- The output space grows **exponentially** with the number of labels
- Labels are often **correlated**
- Evaluation requires **specialised metrics** beyond standard accuracy

In this notebook we focus on **non‑deep‑learning approaches**, using:
- Problem transformation methods
- Algorithm adaptation methods
- Appropriate multi‑label evaluation metrics


## Problem Transformation Methods

Problem transformation methods convert a multi‑label task into one or more **single‑label classification problems**,
allowing us to use standard machine‑learning models.


### Binary Relevance (BR)

**Idea:** Train **one binary classifier per label**.

Each classifier predicts whether a given label should be assigned,
independently of all other labels.

**Pros**
- Simple and scalable

**Cons**
- Ignores label correlations


### Classifier Chains (CC)

Classifier Chains extend Binary Relevance by **feeding predictions of previous labels**
into subsequent classifiers.

This allows the model to learn label dependencies such as:

> *If Politics = 1, Economy becomes more likely*


### Label Powerset (LP)

**Idea:** Treat each unique label combination as a **single class** in a multi‑class problem.

**Pros**
- Captures label dependencies directly

**Cons**
- Does not scale well when many label combinations exist


## Dummy Dataset (Used Throughout This Notebook)

To ensure all code runs correctly, we will use a **small synthetic dataset**
that mimics a text‑classification problem with three labels:

- Politics
- Economy
- Sports


In [1]:
import numpy as np
import pandas as pd

# -----------------
# Create dummy data
# -----------------
np.random.seed(42)

n_samples = 200
n_features = 10
n_labels = 3

X = pd.DataFrame(
    np.random.randn(n_samples, n_features),
    columns=[f"feature_{i}" for i in range(n_features)]
)

Y = pd.DataFrame(
    np.random.randint(0, 2, size=(n_samples, n_labels)),
    columns=["Politics", "Economy", "Sports"]
)

X.head(), Y.head()

(   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
 0   0.496714  -0.138264   0.647689   1.523030  -0.234153  -0.234137   
 1  -0.463418  -0.465730   0.241962  -1.913280  -1.724918  -0.562288   
 2   1.465649  -0.225776   0.067528  -1.424748  -0.544383   0.110923   
 3  -0.601707   1.852278  -0.013497  -1.057711   0.822545  -1.220844   
 4   0.738467   0.171368  -0.115648  -0.301104  -1.478522  -0.719844   
 
    feature_6  feature_7  feature_8  feature_9  
 0   1.579213   0.767435  -0.469474   0.542560  
 1  -1.012831   0.314247  -0.908024  -1.412304  
 2  -1.150994   0.375698  -0.600639  -0.291694  
 3   0.208864  -1.959670  -1.328186   0.196861  
 4  -0.460639   1.057122   0.343618  -1.763040  ,
    Politics  Economy  Sports
 0         1        0       1
 1         0        0       1
 2         0        1       1
 3         0        0       1
 4         1        0       0)

## Train / Test Split

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.25, random_state=42
)

print(X_train.shape, X_test.shape)
print(Y_train.shape, Y_test.shape)

(150, 10) (50, 10)
(150, 3) (50, 3)


## Algorithm Adaptation Methods

Instead of transforming the problem, **algorithm adaptation methods**
modify learning algorithms so they can **directly predict multiple labels**.


### Multi‑Label k‑Nearest Neighbours (ML‑kNN)

ML‑kNN extends kNN by estimating **posterior probabilities per label**
based on the label frequency among nearest neighbours.


In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, jaccard_score, f1_score

knn = KNeighborsClassifier(n_neighbors=5)

multi_knn = MultiOutputClassifier(knn)
multi_knn.fit(X_train, Y_train)

preds = multi_knn.predict(X_test)

print("Hamming Loss:", hamming_loss(Y_test, preds))
print("Jaccard Similarity:", jaccard_score(Y_test, preds, average="samples"))
print("Macro F1:", f1_score(Y_test, preds, average="macro"))


Hamming Loss: 0.5133333333333333
Jaccard Similarity: 0.29
Macro F1: 0.483614575690878


/Users/guywinfield/PycharmProjects/.turing_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Multi‑Output Decision Trees / Random Forests

Standard tree‑based models can be adapted to multi‑label tasks
using a **MultiOutputClassifier** wrapper.


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

rf = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, random_state=42)
)

rf.fit(X_train, Y_train)
preds = rf.predict(X_test)

print("Hamming Loss:", hamming_loss(Y_test, preds))
print("Jaccard Similarity:", jaccard_score(Y_test, preds, average="samples"))
print("Macro F1:", f1_score(Y_test, preds, average="macro"))

Hamming Loss: 0.5333333333333333
Jaccard Similarity: 0.2766666666666666
Macro F1: 0.45803145045842486


/Users/guywinfield/PycharmProjects/.turing_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Multi‑Label Logistic Regression

Logistic regression can be extended to multi‑label settings
by training **one sigmoid per label**.


In [9]:
from sklearn.linear_model import LogisticRegression

logreg = MultiOutputClassifier(
    LogisticRegression(max_iter=1000)
)

logreg.fit(X_train, Y_train)
preds = logreg.predict(X_test)

print("Hamming Loss:", hamming_loss(Y_test, preds))
print("Jaccard Similarity:", jaccard_score(Y_test, preds, average="samples"))
print("Macro F1:", f1_score(Y_test, preds, average="macro"))

Hamming Loss: 0.47333333333333333
Jaccard Similarity: 0.37
Macro F1: 0.5406693577425284


/Users/guywinfield/PycharmProjects/.turing_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Threshold Optimisation (Macro‑F1)

Instead of using the default 0.5 threshold,
we can optimise thresholds to **improve macro‑F1**,
which treats rare and frequent labels equally.


In [10]:
import numpy as np

probs = np.stack(
    [est.predict_proba(X_test)[:, 1] for est in logreg.estimators_],
    axis=1
)

thresholds = np.linspace(0.1, 0.9, 9)

best_f1 = 0
best_preds = None
best_t = None

for t in thresholds:
    preds = (probs >= t).astype(int)
    score = f1_score(Y_test, preds, average="macro")
    if score > best_f1:
        best_f1 = score
        best_preds = preds
        best_t = t

print(f"Best Threshold: {best_t}")
print("Optimised Macro F1:", best_f1)
print("Hamming Loss:", hamming_loss(Y_test, best_preds))
print("Jaccard Similarity:", jaccard_score(Y_test, best_preds, average='samples'))

Best Threshold: 0.1
Optimised Macro F1: 0.6590195077963795
Hamming Loss: 0.5066666666666667
Jaccard Similarity: 0.4933333333333333


## Summary

In this notebook we:

- Introduced **multi‑label classification**
- Covered **problem transformation** and **algorithm adaptation** methods
- Implemented **ML‑kNN**, **Random Forest**, and **Logistic Regression**
- Used **Hamming Loss**, **Jaccard similarity**, and **Macro‑F1**
- Demonstrated **threshold optimisation**

This provides a complete, runnable foundation for real‑world multi‑label problems.
